# 02i — artifact ablation, the instrument's own half

Registered robustness check. Estimated **~15 min** on a T4. Suggested kernel: `emocap-ablation`.

§4.1: *"Accuracy on punctuation-stripped, lowercased captions reported alongside raw. A
large gap means the classifier reads punctuation, and the stripped number becomes primary."*

## Why this is being redone

`runs/classifier/report.json` reports a gap of **exactly 0.0000** — 0.7619 raw against
0.7619 stripped. That is not a clean result, it is **a test that could not fail**: it ran on
the TF-IDF baseline, whose tokenizer discards punctuation before it ever sees the text, so
stripping punctuation changed the input by nothing.

The registered check has therefore never been performed on the frozen DistilRoBERTa, which
*does* see punctuation and produces every number in the study.

## The half already done, and the half this notebook runs

| level | status | result |
|---|---|---|
| **arms** — frozen classifier on each arm's generated captions | done locally | stripping changes 100% of captions; gaps −0.0009 to +0.0123, except `V1_paired5` at **+0.0381** |
| **instrument** — the classifier's own 5-fold CV on stripped text | **this notebook** | compared against the raw CV of **0.8279** recorded when it was frozen |

The arms level is inference and ran in an hour on a laptop. This level retrains
DistilRoBERTa five times, which is the kind of sustained GPU work that wedged a local job
overnight, so it belongs here.

**These two can disagree, and that would be the finding.** If the arms' accuracies survive
stripping but the instrument's CV does not, the classifier leans on punctuation in general
while the arms happen not to exploit it. That distinction belongs in the paper rather than
averaged away.

**Nothing is retrained into the frozen instrument.** The CV models are transient;
`models/register_classifier` is not touched and its sha256 does not change — this notebook
never even loads it.

## Setup

**Attach both:**

- `emocap-v2-arms` — the 13,170-cell training pool
- `emocap-v2-predictions` — **not used by this run**, attached only because the code commit
  is pinned to its provenance. The arms dataset was pushed before these scripts existed, and
  pinning to it checked out code that did not contain them.

**Accelerator:** GPU **T4 x2**. **Internet:** on (`distilroberta-base` is downloaded).

    kaggle kernels output <owner>/emocap-ablation -p tmp/emocap-ablation \
        --page-size 200 --file-pattern 'artifact_'


In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
DATA = "/kaggle/input/emocap-v2-arms"
PREDS = "/kaggle/input/emocap-v2-predictions/predictions"   # for the commit pin only
OUT = "/kaggle/working/results"


In [ ]:
# Pinned to the PREDICTIONS dataset, for the reason 02h had to be: the arms dataset was
# pushed before these analysis scripts existed, so pinning to it checks out a commit that
# does not contain them, and the failure surfaces as a subprocess "no such file" several
# cells later. Both commits are printed so a mismatch is visible.
import json, subprocess, sys
from pathlib import Path

COMMIT = json.loads(Path(PREDS).parent.joinpath("provenance.json").read_text())["git_commit"]
arms_commit = json.loads(Path(DATA, "provenance.json").read_text())["git_commit"]
print("predictions built at", COMMIT[:12], "  <- code pinned here")
print("arms built at       ", arms_commit[:12],
      "" if arms_commit == COMMIT else "  (older, fine -- the arms have not changed)")

if not Path("/kaggle/working/EmoCap").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/asjad2401/EmoCap.git",
                    "/kaggle/working/EmoCap"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/EmoCap", "checkout", "-q", COMMIT], check=True)

script = Path("/kaggle/working/EmoCap/scripts/artifact_ablation.py")
if not script.exists():
    raise SystemExit(f"{script} is absent at {COMMIT[:12]}. The predictions dataset was "
                     f"pushed from a commit that predates it; re-push it.")
print("code at", COMMIT[:12], "and artifact_ablation.py is present")
# The script must also ACCEPT the flags this notebook passes. Checking only that the file
# exists let a stale pin through once: the commit had artifact_ablation.py but not its
# --data-root flag, and argparse rejected the call after the clone had succeeded. One
# --help call catches that in a second.
need = ["--data-root", "--baseline-report", "--skip-arms", "--out"]
helptext = subprocess.run([sys.executable, str(script), "--help"],
                          capture_output=True, text=True).stdout
missing = [f for f in need if f not in helptext]
if missing:
    raise SystemExit(
        f"{script.name} at {COMMIT[:12]} does not accept {missing}.\n"
        f"The predictions dataset was pushed from a commit that predates those flags.\n"
        f"Re-push it: uv run python scripts/push_kaggle.py --part predictions")
print("accepts:", ", ".join(need))


In [ ]:
sys.path.insert(0, "/kaggle/working/EmoCap/src")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# The raw CV accuracy this run is compared against travels with the repo, not the dataset.
baseline = Path("/kaggle/working/EmoCap/runs/classifier/report.json")
if not baseline.exists():
    raise SystemExit(f"{baseline} is missing -- without it there is nothing to compare the "
                     f"stripped accuracy against.")
raw_cv = json.loads(baseline.read_text())["cv"]["accuracy"]
print(f"instrument's RAW 5-fold CV, recorded when it was frozen: {raw_cv}")

for arm in ("S_unpaired", "V1_unpaired", "H_unpaired"):
    p = Path(DATA, "arms", f"{arm}.jsonl")
    if not p.exists():
        raise SystemExit(f"missing {p} -- attach emocap-v2-arms")
print("training pool present")


In [ ]:
import time
Path(OUT).mkdir(parents=True, exist_ok=True)

# --skip-arms: the arms level is inference over 359k captions and was already run locally.
# Redoing it here would add an hour for numbers that are in results/artifact_ablation.json.
cmd = [sys.executable, "-u", "/kaggle/working/EmoCap/scripts/artifact_ablation.py",
       "--skip-arms",
       "--data-root", DATA,
       "--baseline-report", str(baseline),
       "--out", f"{OUT}/artifact_ablation_instrument.json"]

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print("   ", line, end="")
rc = proc.wait()
print(f"\nexit {rc}   [{(time.time()-t0)/60:.1f} min]")
if rc != 0:
    raise RuntimeError(f"artifact_ablation.py failed with rc={rc}")


In [ ]:
r = json.loads(Path(OUT, "artifact_ablation_instrument.json").read_text())["instrument"]
print(f"raw CV       {r['raw_cv_accuracy']:.4f}")
print(f"stripped CV  {r['stripped_cv_accuracy']:.4f}   (n={r['n']:,})")
print(f"gap          {r['gap']:+.4f}")
print(f"stripping changed {r['changed_by_strip']:.1%} of the pool")
print()
if abs(r["gap"]) < 0.02:
    print("The instrument does NOT lean on punctuation: the raw accuracy stands as primary.")
else:
    print("A gap this size means the classifier reads formatting. Per S4.1 the STRIPPED "
          "number becomes primary, and every accuracy in the study is affected.")
print("\nPair this with the arms level in results/artifact_ablation.json. If the arms "
      "survive stripping and the instrument does not, the classifier leans on punctuation "
      "in general while the arms happen not to exploit it -- report both, do not average.")
